# Barrier reconstruction with fewer measurements

Paired resolution experiment: ten distinct Wong parameter samples from
`res_0.25MeV_noise0.01.h5`, selected with seed 42, using noise realization 0.
Each coarser curve retains every k-th original measurement, including both endpoints.
Resolutions: 0.25, 0.5, 1, 1.5, 2, 2.5, 3 MeV (121, 61, 31, 21, 16, 13, 11 points).
The coarser grids are subsets of the original; they are not all nested in one another.

From the repository root, submit:
```bash
sbatch slurm/paper-scripts/3-resolution-submit.slurm
```
This launches 70 BNN fits, seven GP tasks of ten fits each, and a dependent merge/scoring job.
Committed compact metrics allow plotting immediately; retraining is optional.
After retraining, wait for the final job before regenerating metrics.
The dataset manifest records the source SHA256 and selected parameter/row indices.

Settings follow the current PaperFigures log-space GP baseline and the paper BNN scripts:
RBF GP (20 restarts, 500 samples); sum-of-changepoints BNN (width 50, four chains,
1000 warmup, 100 draws per chain, physical-energy D3 lambda 1 on the log target).
Training seed is 0; BNN output retains up to 100 posterior curves as in the existing trainer.


In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'scripts/resolution_experiment.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
from resolution_experiment import summarize, DEFAULT_DATA, DEFAULT_RESULTS
DATA_DIR = Path(os.environ.get('DATA_DIR', DEFAULT_DATA))
RESULT_DIR = Path(os.environ.get('RESULT_DIR', DEFAULT_RESULTS))
PLOT_DIR = ROOT / 'plots/resolution_experiment'
PLOT_DIR.mkdir(parents=True, exist_ok=True)


## Scoring convention: alignment with PaperFigures

MAE is computed from the **pointwise posterior median** for both models, then
normalized **per curve** by dividing by $R_0^2/(\hbar\omega_0)$ before averaging
over the ten curves. The label is copied verbatim from the paper violin plots.
This is not division by peak height or normalization after averaging.

GP barriers and Wong truth use the paper's numerical derivative helper on the
masked grid. BNN barriers use saved autodiff samples. The three-point estimate
uses analytic Wong truth at its native points; differentiating the truth with the
same coarse three-point stencil would conceal the stencil's bias.

**Evaluation intervals:** `INTERVAL_MODE='interior'`, `INSET_MEV=0.0` matches
PaperFigures' current `interior_metrics=True`, `inset_mev=0.0`: model scores use
[Emin, Emax], with no extrapolation. Setting a positive inset trims that amount
from both ends; keep it equal to PaperFigures' inset when comparing figures.
Three-point estimates use only native stencil points within this requested range;
their support cannot include the endpoints. The audit CSV records actual bounds.
Legacy `paper` mode uses the original padded CSV bounds; `common` uses a 3 MeV
inset. Neither is the default. All conventions for normalized median MAE and
coverage aggregation match the current PaperFigures calculations.

**Coverage:** match PaperFigures by averaging empirical coverage across curves
at each level first, then taking the mean absolute deviation from nominal levels
0%, 25%, 50%, 68%, 90%, 95% (100% excluded). This scalar summarizes the paper's
coverage plot. Per-curve absolute coverage deviations are also retained in the CSV.
The three-point intervals use the paper's Gaussian uncertainty prescription.
No curves are filtered, matching `load_matched_pair(... apply_quality_filters=False)`.

The default plots load compact exported metrics without HDF5 access. For an
unexported interval, metrics are recomputed from saved predictions without training. New CSVs are
written to `paper_metrics_interior_inset0MeV/` (with the configured inset in the name), preserving prior scores.


In [ ]:
import score_resolution_paper
from importlib import reload
reload(score_resolution_paper)
from score_resolution_paper import score_saved_fits, PAPER_MAE_LABEL
INTERVAL_MODE = 'interior'
INSET_MEV = 0.0  # Matches PaperFigures SETTINGS['inset_mev']: measured energy range
from paper_plot_data import load_resolution_records, load_resolution_summary
paper_per_curve = paper_summary = None
if INTERVAL_MODE == 'interior':
    paper_per_curve = load_resolution_records(ROOT / 'data/paper_plot_metrics', INSET_MEV)
    paper_summary = load_resolution_summary(ROOT / 'data/paper_plot_metrics', INSET_MEV)
if paper_per_curve is None or paper_summary is None:
    paper_per_curve, paper_summary = score_saved_fits(DATA_DIR, RESULT_DIR, interval=INTERVAL_MODE, inset_mev=INSET_MEV)
per_curve = paper_per_curve[paper_per_curve.target_interval_MeV == 0].copy()
summary = paper_summary[paper_summary.target_interval_MeV == 0].copy()
METRIC_TAG = (f'paper_metrics_interior_inset{INSET_MEV:g}MeV' if INTERVAL_MODE == 'interior'
              else 'paper_metrics_' + INTERVAL_MODE)
PLOT_DIR = ROOT / 'plots/resolution_experiment' / METRIC_TAG
PLOT_DIR.mkdir(parents=True, exist_ok=True)
display(summary[['resolution', 'method', 'barrier_MAE', 'coverage_MAE', 'n_curves']])


In [ ]:
COLORS = {'BNN': 'green', 'GP': 'blue', '3-point': 'black'}
plt.rcParams.update({'font.family': 'serif', 'font.size': 13})
for metric, ylabel, filename in [
    ('barrier_MAE', PAPER_MAE_LABEL, 'barrier_mae'),
    ('coverage_MAE', 'Mean absolute coverage deviation', 'coverage_mae'),
]:
    fig, ax = plt.subplots(figsize=(7, 4.5), layout='constrained')
    for method, color in COLORS.items():
        rows = summary[summary.method == method].sort_values('resolution')
        ax.plot(rows.resolution, rows[metric], 'o-', color=color, label=method)
    ax.set(xlabel=r'$\Delta E$ (MeV)', ylabel=ylabel)
    ax.set_xticks(sorted(summary.resolution.unique()))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    for extension in ('png', 'pdf'):
        fig.savefig(PLOT_DIR / f'{filename}.{extension}', dpi=300)
    plt.show()


Each marker averages ten per-curve normalized median MAEs, or summarizes the
mean empirical-coverage curve. Lines guide the eye. Prior raw/mean-based scores
are preserved separately; the figures in `paper_metrics_interior_inset0MeV/` use the corrected
paper conventions described above.


## Three-point estimates with approximately fixed physical intervals

The draft's interval is the distance from the center to either outer stencil
point; the total stencil width is twice this interval. Targets are 2 and 3 MeV.
An exact target is not always possible using the available measurements. Choose
the nearest positive integer grid offset, with ties selecting the smaller one.
No interpolation or measurements from a finer dataset are introduced.

The table reports actual stencil spacings and whether they exactly match the
target. These spacings are NOT constant across every resolution. Both targets
use 2 MeV on the 2 MeV grid, 2.5 MeV on the 2.5 MeV grid, and 3 MeV on the 3 MeV
grid. All calculations use the scoring conventions and interval mode above.

Each plot overlays both targets, with GP/BNN shown once. Black squares denote the
2 MeV target; open triangles denote the 3 MeV target. Run setup and scoring cells
before this section. Scoring reuses saved HDF5 predictions and needs no SLURM.


In [ ]:
from resolution_experiment import RESOLUTIONS
TARGET_INTERVALS = (2.0, 3.0)
interval_summary = paper_summary[paper_summary.target_interval_MeV.isin(TARGET_INTERVALS)].copy()
interval_metrics = paper_per_curve[(paper_per_curve.method == '3-point') &
                                  paper_per_curve.target_interval_MeV.isin(TARGET_INTERVALS)].copy()
interval_table = interval_metrics.groupby(
    ['target_interval_MeV', 'resolution'], as_index=False
).agg(stencil_offset=('stencil_offset', 'first'),
      actual_interval_MeV=('actual_interval_MeV', 'first'), n_eval=('n_eval', 'first'))
interval_table['exact_target'] = np.isclose(interval_table.target_interval_MeV,
                                          interval_table.actual_interval_MeV)
display(interval_table)


In [ ]:
import scienceplots
from matplotlib.ticker import AutoMinorLocator, MaxNLocator

# Match PaperFigures typography, blue/green palette, and line/tick treatment.
# Mathtext keeps these exports independent of a local LaTeX installation.
PAPER_STYLE = {
    'font.family': 'serif', 'font.size': 14,
    'axes.labelsize': 24, 'axes.titlesize': 30,
    'xtick.labelsize': 20, 'ytick.labelsize': 20,
    'legend.fontsize': 14, 'mathtext.fontset': 'cm', 'axes.linewidth': 1.2,
}
with plt.style.context(['science', 'notebook', 'ieee', 'grid', 'no-latex']), \
     plt.rc_context(PAPER_STYLE):
    for metric, ylabel, filename in [
        ('barrier_MAE', PAPER_MAE_LABEL, 'barrier_mae_fixed_intervals'),
        ('coverage_MAE', 'Mean absolute coverage deviation', 'coverage_mae_fixed_intervals'),
    ]:
        fig, ax = plt.subplots(figsize=(8, 5.5), layout='constrained')
        # Model scores are identical in the two target-interval comparisons.
        for method, label, color, linestyle, marker in [
            ('BNN', 'AutoBNN', '#2A7F62', '--', 'o'),
            ('GP', 'GP', '#3B5BA5', '-.', 'D'),
        ]:
            rows = interval_summary[(interval_summary.target_interval_MeV == TARGET_INTERVALS[0]) &
                                    (interval_summary.method == method)].sort_values('resolution')
            ax.plot(rows.resolution, rows[metric], label=label, color=color,
                    ls=linestyle, lw=2.5, marker=marker, ms=7, mew=1.2, zorder=4)
        # Keep three-point estimates black; distinguish intervals by line/marker.
        # An open larger marker keeps coincident coarse-grid estimates visible.
        for target, linestyle, marker, face, size in [
            (2.0, '-', 's', '#303030', 6),
            (3.0, ':', '^', 'none', 9),
        ]:
            rows = interval_summary[(interval_summary.target_interval_MeV == target) &
                                    (interval_summary.method == '3-point')].sort_values('resolution')
            ax.plot(rows.resolution, rows[metric], color='#303030',
                    ls=linestyle, lw=2, marker=marker, ms=size, mfc=face, mew=1.3,
                    label=rf'3-point, $h\approx {target:g}$ MeV', zorder=3)
        ax.set_xlabel(r'$\Delta E$ (MeV)', labelpad=10)
        ax.set_ylabel(ylabel, labelpad=10)
        ax.set_xticks(RESOLUTIONS)
        ax.set_xticklabels(['0.25', '0.5', '1', '1.5', '2', '2.5', '3'])
        ax.set_xlim(0.12, 3.13)
        ax.set_ylim(0, interval_summary[metric].max() * 1.15)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax.tick_params(axis='both', which='major', direction='in', top=True,
                       right=True, width=1.2, length=6, pad=6)
        ax.tick_params(axis='both', which='minor', direction='in', top=True,
                       right=True, width=1, length=3)
        ax.grid(False, which='both')
        ax.grid(True, which='major', color='gray', alpha=0.25, lw=0.7, ls='--')
        # Above the axes so the legend never obscures a curve.
        ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
                  ncol=2, frameon=False, borderaxespad=0, handlelength=2.8)
        for extension in ('png', 'pdf'):
            fig.savefig(PLOT_DIR / f'{filename}.{extension}', dpi=300, bbox_inches='tight')
        plt.show()
